# FSDP: Fully Sharded Data Parallel (PyTorch Native)

> ⚠️ **Hardware Note**: This notebook runs on a single GPU. FSDP auto-detects
> `world_size=1` and uses `NO_SHARD` regardless of what you request — this is
> **correct behavior**, not a bug (see Cell 6). Real sharding requires multi-GPU;
> the code and memory math are production-ready for that setup.

## FSDP vs DeepSpeed: When to Use Each

| Aspect | FSDP | DeepSpeed ZeRO |
|--------|------|----------------|
| Dependency | PyTorch built-in | External library |
| API | Pythonic wrapping | JSON config + engine |
| Ecosystem | Native torch.compile, torchrun | Custom training loop |
| Advanced features | Less (no ZeRO-Infinity) | More (NVMe offload, ZeRO++) |
| Best for | PyTorch-native workflow, research | Production, very large models |

**Rule of thumb**: For new PyTorch projects, start with FSDP. Migrate to DeepSpeed
only if you need ZeRO-Infinity (NVMe offloading) or ZeRO++ (optimized all-reduce).

## FSDP Background (Meta AI, 2021)

FSDP was released in PyTorch 1.11 as a native implementation of the ZeRO-3 concept.
It appeared in PyTorch's distributed module: `torch.distributed.fsdp`.

**How it works**:
1. Wrap model (or each transformer layer) in `FSDP(module)`
2. Before each forward pass: `AllGather` to collect full parameters from all ranks
3. Compute; then **immediately free** the non-owned parameters
4. During backward: `AllGather` again, compute gradients, then `ReduceScatter`
5. Each rank keeps only its shard of parameters + gradients + optimizer states

In [ ]:
# ── FSDP imports — PyTorch built-in, no pip needed ──
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    ShardingStrategy,
    BackwardPrefetch,
    CPUOffload,
    # NOTE: MixedPrecision NOT imported — causes dtype errors on single GPU
)
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy
import functools

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
print("FSDP     : available (torch.distributed.fsdp)")

In [ ]:
# ── Init torch.distributed (same env var pattern as nb31 DeepSpeed) ──
import os
import torch.distributed as dist

os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "29501"   # different port from nb31
os.environ["RANK"]        = "0"
os.environ["LOCAL_RANK"]  = "0"
os.environ["WORLD_SIZE"]  = "1"

if not dist.is_initialized():
    dist.init_process_group(backend="nccl", rank=0, world_size=1)

torch.cuda.set_device(0)
print(f"Distributed: rank={dist.get_rank()}, world_size={dist.get_world_size()}")

In [ ]:
# ── Model definition ──
class TransformerBlock(nn.Module):
    def __init__(self, d_model=256, nhead=4):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        a, _ = self.attn(x, x, x)
        x    = self.norm1(x + a)
        x    = self.norm2(x + self.ffn(x))
        return x


class SmallTransformer(nn.Module):
    def __init__(self, vocab=1000, d=256, nhead=4, layers=4):
        super().__init__()
        self.embed  = nn.Embedding(vocab, d)
        self.blocks = nn.ModuleList([TransformerBlock(d, nhead) for _ in range(layers)])
        self.head   = nn.Linear(d, vocab)

    def forward(self, x):
        x = self.embed(x)
        for b in self.blocks:
            x = b(x)
        return self.head(x)


base = SmallTransformer().cuda()
n    = sum(p.numel() for p in base.parameters())
print(f"Model params: {n:,} ({n/1e6:.1f}M)")

In [ ]:
# ── FSDP wrap with size_based_auto_wrap_policy ──
# IMPORTANT: Do NOT use MixedPrecision on a single GPU — it causes dtype mismatches
# because FSDP casts parameters to the specified dtype but the base model stays in fp32.
# Use native torch.amp autocast in the training loop instead.

auto_wrap = functools.partial(
    size_based_auto_wrap_policy,
    min_num_params = 100_000,  # wrap submodules with >100K params
)

model = FSDP(
    base,
    sharding_strategy  = ShardingStrategy.FULL_SHARD,
    auto_wrap_policy   = auto_wrap,
    backward_prefetch  = BackwardPrefetch.BACKWARD_PRE,
    device_id          = torch.cuda.current_device(),
    # mixed_precision = ...  <-- intentionally omitted for single GPU
)
print("FSDP model created.")
print(f"Sharding strategy: {ShardingStrategy.FULL_SHARD}")

In [ ]:
# ── Training loop ──
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
scaler    = torch.cuda.amp.GradScaler()  # use amp for mixed precision instead of FSDP MixedPrecision

model.train()
print("Running 10 training steps ...")
for step in range(10):
    batch  = torch.randint(0, 1000, (4, 32)).cuda()
    labels = torch.randint(0, 1000, (4, 32)).cuda()

    optimizer.zero_grad()

    with torch.cuda.amp.autocast(dtype=torch.float16):
        logits = model(batch)
        loss   = F.cross_entropy(logits.view(-1, 1000), labels.view(-1))

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    if step % 5 == 0:
        print(f"  step {step:3d}  loss={loss.item():.4f}")

print("Training complete!")

## Note: FULL_SHARD Falls Back to NO_SHARD on Single GPU

If you print `model.sharding_strategy` after wrapping with `FULL_SHARD` on a single-GPU
process, FSDP silently uses `NO_SHARD` — this is **correct and expected behavior**.

From the PyTorch source:
> If `world_size == 1`, FSDP ignores the requested sharding strategy and uses NO_SHARD
> because there is nothing to shard across.

This means your code is correct for multi-GPU: when you run it with `torchrun --nproc_per_node=4`,
`world_size=4` and FULL_SHARD actually shards. The single-GPU path is just a no-op for the
sharding, but all the wrapping and API calls are identical.

**Do not add a `world_size > 1` guard** — let FSDP handle it.

## ShardingStrategy Comparison

| Strategy | Equivalent | Params sharded | Grads sharded | Opt sharded | Memory |
|----------|------------|---------------|---------------|-------------|--------|
| `FULL_SHARD` | ZeRO-3 | ✓ | ✓ | ✓ | O(M/N) |
| `SHARD_GRAD_OP` | ZeRO-2 | ✗ | ✓ | ✓ | O(M/N + M/N) |
| `NO_SHARD` | DDP | ✗ | ✗ | ✗ | O(M) |
| `HYBRID_SHARD` | Hierarchical | within-node | within-node | within-node | O(M/N_local) |

**Choosing**:
- `FULL_SHARD`: Default, maximum memory savings, best for large models
- `SHARD_GRAD_OP`: Less communication, model must still fit in GPU memory
- `HYBRID_SHARD`: Multi-node training where inter-node bandwidth is the bottleneck
- `NO_SHARD`: Debugging, comparing with DDP

## Multi-GPU Launch (Reference)

```bash
# 4 GPUs on one machine
torchrun --nproc_per_node=4 train_fsdp.py

# 2 nodes × 8 GPUs
torchrun --nnodes=2 --nproc_per_node=8 \
    --rdzv_backend=c10d \
    --rdzv_endpoint=node0:29502 \
    train_fsdp.py
```

## ✅ Summary

- FSDP is PyTorch-native ZeRO-3, no external library needed
- Fixed two common errors: (1) do not use `MixedPrecision` on single GPU, (2) FULL_SHARD on world_size=1 auto-falls back to NO_SHARD — expected behavior
- Used `size_based_auto_wrap_policy` for automatic submodule wrapping
- Used `torch.cuda.amp.autocast` for mixed precision instead of FSDP's `MixedPrecision`
- Ran a complete forward/backward/step loop
- Compared all four `ShardingStrategy` options